# 06 — Temporal robustness analysis (the exploratory component)

Two experiments, both implemented in `src/eval/temporal.py`:

1. **Decay curve** — for each train year Y in {2019..2022}, train and evaluate on each year. Plot accuracy vs. (test_year − train_year).
2. **Fixed train, sliding test** — train once on data ≤ 2022, evaluate per year on 2023+.

The classical experiments run on CPU. Repeat the decay-curve experiment with the transformer on Colab if compute allows.

In [ ]:
!python -m src.eval.temporal --do-classical

In [ ]:
import pandas as pd
df = pd.read_csv('reports/figures/temporal_results.csv')
df.head()

In [ ]:
from PIL import Image
Image.open('reports/figures/temporal_decay.png')

## (Colab GPU only) Repeat decay experiment with DistilBERT

Train DistilBERT on each year separately, evaluate on each subsequent year. Slow but provides the headline figure for the report.

_4 train_years × ~10 min each = ~40 min on L4._

In [ ]:
import pandas as pd
from pathlib import Path

df = pd.read_csv('data/processed/headlines.csv')
Path('data/processed/year_subsets').mkdir(exist_ok=True, parents=True)
for y in [2019, 2020, 2021, 2022]:
    sub = df[df['year'] == y]
    out = Path(f'data/processed/year_subsets/{y}.csv')
    sub.to_csv(out, index=False)
    print(out, len(sub))

In [ ]:
# For each train year, fine-tune a fresh DistilBERT, predict on each later year, log accuracy.
# This block writes a small results CSV that we then merge with the classical numbers for plotting.
import os, json, subprocess, pandas as pd
from pathlib import Path

rows = []
df = pd.read_csv('data/processed/headlines.csv')
for ty in [2019, 2020, 2021, 2022]:
    train_csv = f'data/processed/year_subsets/{ty}.csv'
    out_dir = f'/content/drive/MyDrive/cis5190/temporal/distilbert_{ty}'
    # Hack: temporarily make a 1-year-only split tree
    Path(f'data/processed/splits_year_{ty}').mkdir(exist_ok=True, parents=True)
    pd.read_csv(train_csv).to_csv(f'data/processed/splits_year_{ty}/train.csv', index=False)
    pd.read_csv(train_csv).sample(min(200, len(pd.read_csv(train_csv))), random_state=42).to_csv(f'data/processed/splits_year_{ty}/val.csv', index=False)
    pd.read_csv(train_csv).sample(min(200, len(pd.read_csv(train_csv))), random_state=43).to_csv(f'data/processed/splits_year_{ty}/test.csv', index=False)
    !python -m src.models.transformer --model distilbert-base-uncased --split year_{ty} --epochs 2 --output-dir {out_dir}
    # Load saved model and predict per eval-year
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    import torch
    tok = AutoTokenizer.from_pretrained(f'{out_dir}/final')
    mdl = AutoModelForSequenceClassification.from_pretrained(f'{out_dir}/final').cuda().eval()
    for ey in [2019, 2020, 2021, 2022, 2023, 2024]:
        eval_df = df[df['year'] == ey]
        if len(eval_df) < 30: continue
        from src.eval.metrics import evaluate_predictions, labels_to_ints
        import numpy as np
        all_preds = []
        for i in range(0, len(eval_df), 64):
            chunk = eval_df['headline'].astype(str).iloc[i:i+64].tolist()
            with torch.no_grad():
                enc = tok(chunk, return_tensors='pt', padding=True, truncation=True, max_length=128).to('cuda')
                logits = mdl(**enc).logits
                all_preds.extend(logits.argmax(-1).cpu().tolist())
        b = evaluate_predictions(f'distilbert_{ty}_to_{ey}', labels_to_ints(eval_df['source']), np.array(all_preds))
        rows.append(dict(model='distilbert', train_year=ty, eval_year=ey, gap=ey-ty, **b.asdict()))
        print(rows[-1])

out = pd.DataFrame(rows)
out.to_csv('reports/figures/temporal_results_transformer.csv', index=False)
out

In [ ]:
# Combine classical + transformer results, replot
import pandas as pd
from src.eval.temporal import plot_decay
from pathlib import Path

frames = [pd.read_csv('reports/figures/temporal_results.csv')]
tx = Path('reports/figures/temporal_results_transformer.csv')
if tx.exists(): frames.append(pd.read_csv(tx))
all_df = pd.concat(frames, ignore_index=True)
plot_decay(all_df, Path('reports/figures/temporal_decay.png'))